In [ ]:
# Cell 1
!pip install numpy==1.26.4 -q
!pip install torch-geometric -q

In [ ]:
# Cell 2
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3
from google.colab import files
files.upload()  # benchmark_master.tsv

Saving benchmark_master.tsv to benchmark_master (1).tsv


{'benchmark_master (1).tsv': b"pdb\tdate\tresolution\tHchain\tLchain\tantigen_chain\tantigen_name\tantigen_species\thas_light\thseq\tlseq\tcdr_h1\tcdr_h2\tcdr_h3\tcdr_h3_len\tcdr_l1\tcdr_l2\tcdr_l3\tcdr_l3_len\n7y3o\t12/27/23\t2.1\tH\tL\tA\tspike protein s1\tsevere acute respiratory syndrome coronavirus2 \tTrue\tEVQLVESGGGVVQPGGSLRLSCAVSGFTFDDYAMHWVRQAPGKGLDWVSLISGDGSYTYYADSVKGRFTISRDSSKNSLYLQMNSLRTEDTALYYCAKAQTPTLWWLQDAFDIWGQGTMVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKS\tDIQMTQSPSSLSASVGDRVTITCRASQSISNYLNWYQQKPGKAPKLLIYVASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQSYSTPFTFGPGTKVDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC\tGFTFDDYA\tISGDGSYT\tAKAQTPTLWWLQDAFDI\t17\tQSISNY\tVAS\tQQSYSTPFT\t9.0\n7yds\t07/19/23\t2.3\tB\tC\tA\tprogrammed cell death 1 ligand 1\thomo sapiens\tTrue\tDVQLVESGGGLVQPGGSLRLSCTVSGIDLSSYDMTWVRQAPGKGLEYIGYISYVSRTYYADSVKGRFTISKDTSKNTVYLQMNSLR

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool
import numpy as np
import pandas as pd
import os
import glob
from scipy import stats
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

DRIVE = '/content/drive/MyDrive/antibody_benchmark'
RESULTS_DIR = f'{DRIVE}/paegat_v1_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

BOLTZ_DIR = f'{DRIVE}/paegat_dataset'
OF3_DIR = f'{DRIVE}/of3_paegat_dataset'
PROT_DIR = f'{DRIVE}/protenix_paegat_dataset'

def load_graphs(dataset_dir):
    files = glob.glob(f'{dataset_dir}/*.pt')
    graphs = []
    for f in files:
        try:
            graphs.append(torch.load(f, weights_only=False))
        except:
            pass
    print(f"Loaded {len(graphs)} from {dataset_dir.split('/')[-1]}")
    return graphs

print("Loading datasets...")
boltz_graphs = load_graphs(BOLTZ_DIR)
of3_graphs = load_graphs(OF3_DIR)
prot_graphs = load_graphs(PROT_DIR)

# After loading all datasets, normalize attributes
for g in boltz_graphs + of3_graphs + prot_graphs:
    if hasattr(g, 'seed_dir'):
        del g.seed_dir
    if hasattr(g, 'seed'):
        del g.seed
print(f"Total: Boltz-2={len(boltz_graphs)}, OF3={len(of3_graphs)}, Protenix={len(prot_graphs)}")

class PAEGAT(nn.Module):
    def __init__(self, in_channels=23, hidden_channels=64, heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        self.conv1 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels // heads, heads=heads, edge_dim=1, dropout=dropout)
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr, batch):
        x = F.relu(self.input_proj(x))
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = global_mean_pool(x, batch)
        return self.output(x).squeeze(-1)

def compute_ece(confidence, accuracy, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(confidence[mask].mean() - accuracy[mask].mean())
    return ece

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    preds, labels, iptms, pdbs = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            preds.extend(pred.cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
            iptms.extend(batch.iptm.cpu().numpy())
            pdbs.extend(batch.pdb if isinstance(batch.pdb, list) else [batch.pdb])
    return np.array(preds), np.array(labels), np.array(iptms), pdbs


def run_round(round_num, train_graphs, test_graphs, test_model_name):
    print(f"\n{'='*50}")
    print(f"ROUND {round_num}: Train on {len(train_graphs)} graphs, Test on {test_model_name} ({len(test_graphs)} graphs)")
    print(f"{'='*50}")

    train_loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=16, shuffle=False)

    model = PAEGAT().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()

    best_loss = float('inf')
    best_weights = None

    for epoch in range(50):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        scheduler.step(train_loss)

        if train_loss < best_loss:
            best_loss = train_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/50 | Loss: {train_loss:.4f}")

    # Load best weights
    model.load_state_dict(best_weights)
    torch.save(best_weights, f'{RESULTS_DIR}/paegat_v1_round{round_num}.pt')

    # Evaluate
    preds, labels, iptms, pdbs = evaluate(model, test_loader)
    acceptable = (labels >= 0.23).astype(float)

    ece_paegat = compute_ece(preds, acceptable)
    ece_iptm = compute_ece(iptms, acceptable)
    r_paegat, _ = stats.spearmanr(preds, labels)
    r_iptm, _ = stats.spearmanr(iptms, labels)

    high_conf_mask = iptms >= 0.7
    fpr_iptm = ((iptms >= 0.7) & (labels < 0.23)).sum() / max((iptms >= 0.7).sum(), 1)
    high_paegat_mask = preds >= 0.5
    fpr_paegat = ((preds >= 0.5) & (labels < 0.23)).sum() / max((preds >= 0.5).sum(), 1)

    print(f"\nRound {round_num} Results (test on {test_model_name}):")
    print(f"ECE   — PAE-GAT: {ece_paegat:.4f} vs ipTM: {ece_iptm:.4f} | improvement: {((ece_iptm-ece_paegat)/ece_iptm*100):.1f}%")
    print(f"r     — PAE-GAT: {r_paegat:.3f} vs ipTM: {r_iptm:.3f}")
    print(f"FPR   — PAE-GAT: {fpr_paegat:.3f} vs ipTM: {fpr_iptm:.3f}")

    # Save results
    df_results = pd.DataFrame({
        'pdb': pdbs,
        'dockq_ha': labels,
        'paegat_pred': preds,
        'iptm': iptms,
        'test_model': test_model_name,
        'round': round_num
    })
    df_results.to_csv(f'{RESULTS_DIR}/paegat_v1_round{round_num}_results.tsv', sep='\t', index=False)

    return {
        'round': round_num,
        'test_model': test_model_name,
        'ece_paegat': ece_paegat,
        'ece_iptm': ece_iptm,
        'r_paegat': r_paegat,
        'r_iptm': r_iptm,
        'fpr_paegat': fpr_paegat,
        'fpr_iptm': fpr_iptm,
        'ece_improvement_pct': (ece_iptm - ece_paegat) / ece_iptm * 100
    }

# Run all three rounds
summary = []

# Round 1: Train OF3 + Protenix, Test Boltz-2
summary.append(run_round(1, of3_graphs + prot_graphs, boltz_graphs, 'Boltz-2'))

# Round 2: Train Boltz-2 + Protenix, Test OF3
summary.append(run_round(2, boltz_graphs + prot_graphs, of3_graphs, 'OpenFold3'))

# Round 3: Train Boltz-2 + OF3, Test Protenix
summary.append(run_round(3, boltz_graphs + of3_graphs, prot_graphs, 'Protenix'))

# Summary
df_summary = pd.DataFrame(summary)
df_summary.to_csv(f'{RESULTS_DIR}/paegat_v1_summary.tsv', sep='\t', index=False)

print(f"\n{'='*50}")
print("FINAL SUMMARY — PAE-GAT Version 1")
print(f"{'='*50}")
for _, row in df_summary.iterrows():
    print(f"Round {int(row['round'])} (test {row['test_model']}): ECE {row['ece_paegat']:.4f} vs {row['ece_iptm']:.4f} | {row['ece_improvement_pct']:.1f}% improvement")

print(f"\nMean ECE PAE-GAT: {df_summary['ece_paegat'].mean():.4f} ± {df_summary['ece_paegat'].std():.4f}")
print(f"Mean ECE ipTM:    {df_summary['ece_iptm'].mean():.4f} ± {df_summary['ece_iptm'].std():.4f}")
print(f"Mean improvement: {df_summary['ece_improvement_pct'].mean():.1f}%")

# Disconnect runtime to save units
print("\nTraining complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()



Device: cuda
Loading datasets...
Loaded 1109 from paegat_dataset
Loaded 1129 from of3_paegat_dataset
Loaded 1162 from protenix_paegat_dataset
Total: Boltz-2=1109, OF3=1129, Protenix=1162

ROUND 1: Train on 2291 graphs, Test on Boltz-2 (1109 graphs)
Epoch 10/50 | Loss: 0.0360
Epoch 20/50 | Loss: 0.0339
Epoch 30/50 | Loss: 0.0309
Epoch 40/50 | Loss: 0.0287
Epoch 50/50 | Loss: 0.0286

Round 1 Results (test on Boltz-2):
ECE   — PAE-GAT: 0.0478 vs ipTM: 0.3219 | improvement: 85.2%
r     — PAE-GAT: 0.561 vs ipTM: 0.631
FPR   — PAE-GAT: 0.243 vs ipTM: 0.446

ROUND 2: Train on 2271 graphs, Test on OpenFold3 (1129 graphs)
Epoch 10/50 | Loss: 0.0445
Epoch 20/50 | Loss: 0.0401
Epoch 30/50 | Loss: 0.0380
Epoch 40/50 | Loss: 0.0365
Epoch 50/50 | Loss: 0.0344

Round 2 Results (test on OpenFold3):
ECE   — PAE-GAT: 0.1908 vs ipTM: 0.2072 | improvement: 7.9%
r     — PAE-GAT: 0.659 vs ipTM: 0.633
FPR   — PAE-GAT: 0.045 vs ipTM: 0.158

ROUND 3: Train on 2238 graphs, Test on Protenix (1162 graphs)
Epoch 1